## Librerias

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import os
import joblib
from datetime import datetime
from dotenv import load_dotenv
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import classification_report, confusion_matrix
import xgboost as xgb





In [ ]:
pd.set_option('display.max_columns', None)


In [ ]:
load_dotenv()
DB_PATH = os.path.join('..', 'db', 'gaming_warehouse.db')
MODELS_DIR = os.path.join('..', 'models')
os.makedirs(MODELS_DIR, exist_ok=True)


In [23]:
def Query(query):
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql_query(query, conn)
    conn.close()
    return df


In [24]:
print(DB_PATH)

..\db\gaming_warehouse.db


## Carga de datos

In [ ]:
df_hist          = Query('SELECT * FROM Hist_Precios_ITAD')
df_tiendas       = Query('SELECT * FROM CAT_Tienda')
df_juegos        = Query('''
    SELECT j.* FROM CAT_Juego j
    WHERE j.juego_id NOT IN (SELECT juego_id_dlc FROM REL_Juego_DLC)
''')
df_rel_itad      = Query('SELECT * FROM REL_Juego_ITAD')
df_generos       = Query('''
    SELECT r.juego_id, GROUP_CONCAT(g.nombre, '|') AS generos
    FROM REL_Juego_Genero r
    JOIN CAT_Genero g ON r.genero_id = g.genero_id
    GROUP BY r.juego_id
''')
df_modos         = Query('''
    SELECT r.juego_id, GROUP_CONCAT(m.nombre, '|') AS modos
    FROM REL_Juego_Modo r
    JOIN CAT_Modo_Juego m ON r.modo_id = m.modo_id
    GROUP BY r.juego_id
''')
df_perspectivas  = Query('''
    SELECT r.juego_id, GROUP_CONCAT(p.nombre, '|') AS perspectivas
    FROM REL_Juego_Perspectiva r
    JOIN CAT_Perspectiva p ON r.perspectiva_id = p.perspectiva_id
    GROUP BY r.juego_id
''')
df_resenas_agg   = Query('''
    SELECT
        juego_id,
        COUNT(*) AS total_resenas,
        ROUND(100.0 * SUM(recomendado) / COUNT(*), 2) AS pct_positivo,
        ROUND(AVG(puntuacion_ponderada), 4) AS avg_score_ponderado
    FROM Hist_Steam_Reviews
    GROUP BY juego_id
''')

print(df_hist.shape)
print(df_tiendas.shape)
print(df_juegos.shape)
print(df_rel_itad.shape)
print(df_resenas_agg.shape)

hist:         (548468, 5)
tiendas:      (30, 2)
juegos:       (8797, 24)
rel_itad:     (6545, 2)
resenas_agg:  (5205, 4)


## Limpieza de tiendas y pares con poco volumen

In [ ]:
UMBRAL_TIENDA = 10_000
UMBRAL_PAR    = 10

#Filtrar tiendas con pco volmen
tiendas_validas = (df_hist.groupby('id_tienda').size() .reset_index(name='registros')
                   .query('registros >= @UMBRAL_TIENDA')['id_tienda'] .tolist())

df = df_hist[df_hist['id_tienda'].isin(tiendas_validas)].copy()

df['fecha'] = pd.to_datetime(df['fecha_unix'], unit='s')
df = df.sort_values(['itad_id_texto', 'id_tienda', 'fecha']).reset_index(drop=True)

# quitar tiendas-juegos con poco historial
pares_validos = (df.groupby(['itad_id_texto', 'id_tienda']).size().reset_index(name='registros')
                 .query('registros >= @UMBRAL_PAR')[['itad_id_texto', 'id_tienda']])

df = df.merge(pares_validos, on=['itad_id_texto', 'id_tienda'], how='inner').reset_index(drop=True)

print(len(df))
print(df.groupby(["itad_id_texto","id_tienda"]).ngroups)
print(df["id_tienda"].nunique())

Registros: 479,403
Pares:     22,045
Tiendas:   21


### etiquetas

In [ ]:
VENTANA_DIAS = 30

def construir_etiqueta(df):
    df = df.copy()
    df['etiqueta'] = 0
    etiquetas = []

    for (_, _), grupo in df.groupby(['itad_id_texto', 'id_tienda']):
        grupo = grupo.sort_values('fecha').reset_index()
        fechas     = grupo['fecha'].values
        descuentos = grupo['corte_descuento'].values
        indices    = grupo['index'].values

        for i in range(len(grupo)):
            fecha_limite    = fechas[i] + np.timedelta64(VENTANA_DIAS, 'D')
            mask            = (fechas > fechas[i]) & (fechas <= fecha_limite)
            max_desc_futuro = descuentos[mask].max() if mask.any() else 0

            if   max_desc_futuro >= 90: clase = 4
            elif max_desc_futuro >= 75: clase = 3
            elif max_desc_futuro >= 50: clase = 2
            elif max_desc_futuro >= 20: clase = 1
            else:                       clase = 0

            etiquetas.append((indices[i], clase))

    idx, clases = zip(*etiquetas)
    df.loc[list(idx), 'etiqueta'] = list(clases)
    return df

df = construir_etiqueta(df)
df = df.sort_values(['itad_id_texto', 'id_tienda', 'fecha']).reset_index(drop=True)

labels = {0:'<20%', 1:'20-49%', 2:'50-74%', 3:'75-89%', 4:'90%+'}
for c, n in df['etiqueta'].value_counts().sort_index().items():
    print(f'  Clase {c} ({labels[c]}): {n:,} ({100*n/len(df):.1f}%)')

Construyendo etiquetas...
  Clase 0 (<20%): 170,921 (35.7%)
  Clase 1 (20-49%): 43,044 (9.0%)
  Clase 2 (50-74%): 92,756 (19.3%)
  Clase 3 (75-89%): 131,370 (27.4%)
  Clase 4 (90%+): 41,312 (8.6%)


##  ingenireira de materiales

In [ ]:
# variabels de fechas
df['mes']        = df['fecha'].dt.month
df['dia_semana'] = df['fecha'].dt.dayofweek
df['semana_anio'] = df['fecha'].dt.isocalendar().week.astype(int)

def es_periodo_sale(semana):
    return int(semana in list(range(13,16)) + list(range(24,28)) + list(range(46,49)) + [51,52,1])

df['es_periodo_sale'] = df['semana_anio'].apply(es_periodo_sale)

#tenemos un bune descuento?
df['es_descuento'] = (df['corte_descuento'] >= 20).astype(int)

# precio base que se tuvo?
precio_base = (df.groupby(['itad_id_texto','id_tienda'])['precio']
               .max().reset_index().rename(columns={'precio':'precio_base'}))
df = df.merge(precio_base, on=['itad_id_texto','id_tienda'], how='left')
df['precio_vs_base'] = (df['precio'] / df['precio_base']).round(4)

# tenemos un minimo reciente
df['en_minimo_reciente'] = (df['precio_vs_base'] <= 0.19).astype(int)

# descuento maximo y promdedio
df['max_descuento_historico'] = (df
    .groupby(['itad_id_texto','id_tienda'])['corte_descuento']
    .expanding().max()
    .reset_index(level=[0,1], drop=True)
    .shift(1).fillna(0))

df['avg_descuento_historico'] = (df.groupby(['itad_id_texto','id_tienda'])['corte_descuento'].expanding().mean().reset_index(level=[0,1], drop=True)
    .shift(1).fillna(0).round(2))

# mediana de precio
df['precio_mediana_exp'] = (df.groupby(['itad_id_texto','id_tienda'])['precio'].expanding().median()
    .reset_index(level=[0,1], drop=True)
    .shift(1).bfill().fillna(0))

#problemas con inf
df['precio_vs_mediana'] = (df['precio'] / df['precio_mediana_exp']).round(4).replace([np.inf,-np.inf], 1.0)

# avlor y fecha del ultimo descuento bueno
descuento_sig = df['corte_descuento'].where(df['corte_descuento'] >= 20, other=np.nan)
df['ultimo_descuento_pct'] = (df.groupby(['itad_id_texto','id_tienda']).apply(lambda g: g['corte_descuento'].where(g['corte_descuento'] >= 20).shift(1).ffill())
    .reset_index(level=[0,1], drop=True).fillna(0))

fecha_ultimo_desc = df['fecha'].where(df['corte_descuento'] >= 20, other=pd.NaT)
df['fecha_ultimo_desc'] = (df
    .groupby(['itad_id_texto','id_tienda'])['fecha']
    .apply(lambda g: g.where(df.loc[g.index,'corte_descuento'] >= 20).shift(1).ffill())
    .reset_index(level=[0,1], drop=True))

df['dias_desde_ultimo_desc'] = (df['fecha'] - df['fecha_ultimo_desc']).dt.days.fillna(-1).astype(int)
df = df.drop(columns=['fecha_ultimo_desc'])

# descuentos acumulados que ha tenido el juego
df['n_descuentos_acum'] = (df
    .groupby(['itad_id_texto','id_tienda'])['es_descuento']
    .expanding().sum()
    .reset_index(level=[0,1], drop=True)
    .shift(1).fillna(0))

# días en historial
df['dias_en_historial'] = (df
    .groupby(['itad_id_texto','id_tienda'])['fecha']
    .transform(lambda x: (x - x.min()).dt.days + 1))
#frecuencia de descuento en mes
df['frec_descuentos_mes'] = (df['n_descuentos_acum'] / (df['dias_en_historial'] / 30)
                              ).replace([np.inf,-np.inf], 0).round(3)

for lag in [1, 2, 3]:
    df[f'descuento_lag_{lag}'] = (df
        .groupby(['itad_id_texto','id_tienda'])['corte_descuento']
        .shift(lag).fillna(0))
    df[f'precio_lag_{lag}'] = (df
        .groupby(['itad_id_texto','id_tienda'])['precio']
        .shift(lag).bfill().fillna(0))

df['delta_precio'] = (df['precio'] - df['precio_lag_1']).fillna(0)

# tiempo sin descuento
def calcular_racha(serie):
    racha, contador = [], 0
    for val in serie:
        contador = 0 if val >= 20 else contador + 1
        racha.append(contador)
    return racha

df['racha_sin_descuento'] = (df
    .groupby(['itad_id_texto','id_tienda'])['corte_descuento']
    .transform(calcular_racha))

# tienda como feature numérico
df['tienda_id'] = df['id_tienda'].astype('category').cat.codes

print(f'Features de historial agregados. Shape: {df.shape}')
print(f'Nulos: {df.isnull().sum().sum()}')

Features de historial agregados. Shape: (479403, 33)
Nulos: 0


## ingeniera del catálogo

In [ ]:
df_catalogo = (df_rel_itad
               .merge(df_juegos,       on='juego_id', how='left')
               .merge(df_generos,      on='juego_id', how='left')
               .merge(df_modos,        on='juego_id', how='left')
               .merge(df_perspectivas, on='juego_id', how='left')
               .merge(df_resenas_agg,  on='juego_id', how='left'))

df_catalogo['edad_juego_dias'] = (
    pd.Timestamp.now() - pd.to_datetime(df_catalogo['fecha_lanzamiento'])
).dt.days

# OHE, quitar uno para evitar la multicolinelaidad
df_generos_ohe = df_catalogo['generos'].str.get_dummies(sep='|').add_prefix('gen_').iloc[:, 1:]
df_modos_ohe   = df_catalogo['modos'].str.get_dummies(sep='|').add_prefix('modo_').iloc[:, 1:]
df_persp_ohe   = df_catalogo['perspectivas'].str.get_dummies(sep='|').add_prefix('persp_').iloc[:, 1:]

cols_catalogo = [
    'juego_id', 'itad_id_texto','puntuacion_igdb', 'conteo_votos_igdb', 'recommendations_count','metacritic_score', 'edad_juego_dias', 'conteo_dlc',
    'achievements_count', 'total_resenas', 'pct_positivo', 'avg_score_ponderado'
]

df_catalogo = pd.concat(
    [df_catalogo[cols_catalogo], df_generos_ohe, df_modos_ohe, df_persp_ohe],
    axis=1
).drop_duplicates(subset='itad_id_texto')

print(f'Catálogo forma: {df_catalogo.shape}')

Catálogo shape: (6077, 45)


## 7. Unión y limpieza de nulos

In [ ]:
df_modelo = df.merge(df_catalogo.drop(columns=['juego_id']), on='itad_id_texto', how='left')

# imputación con mediana suele dar mejore resultados
cols_mediana = ['puntuacion_igdb','conteo_votos_igdb','edad_juego_dias',
                'conteo_dlc','recommendations_count','achievements_count']
for col in cols_mediana:
    df_modelo[col] = df_modelo[col].fillna(df_modelo[col].median())

df_modelo['tiene_metacritic'] = df_modelo['metacritic_score'].notna().astype(int)
df_modelo['metacritic_score'] = df_modelo['metacritic_score'].fillna(df_modelo['metacritic_score'].median())
df_modelo['total_resenas']= df_modelo['total_resenas'].fillna(0)
df_modelo['pct_positivo']= df_modelo['pct_positivo'].fillna(50.0)
df_modelo['avg_score_ponderado'] = df_modelo['avg_score_ponderado'].fillna(0)

print(df_modelo.shape)
print(df_modelo.isnull().sum().sum())

Shape df_modelo: (479403, 77)
Nulos restantes: 0


## separar datos

In [ ]:
cols_excluir = [
    'itad_id_texto', 'id_tienda', 'precio','corte_descuento','fecha_unix', 'fecha', 'etiqueta', 
    'es_descuento',
    'precio_base', 'precio_mediana_exp'
]

X = df_modelo.drop(columns=[c for c in cols_excluir if c in df_modelo.columns])
y = df_modelo['etiqueta']

#al sere valores de tiempo cortar por fecha
fecha_corte= df_modelo['fecha'].quantile(0.8)
train_mask= df_modelo['fecha'] <= fecha_corte
test_mask= df_modelo['fecha'] > fecha_corte

X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

#tengo dewsbalance en las clases, tocara agregarles peso
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Features: {X_train.shape[1]}')
print(f'Distribución train:')
print(y_train.value_counts().sort_index())

Train: (383528, 67)  |  Test: (95875, 67)
Features: 67
Distribución train:
etiqueta
0    124507
1     31746
2     78476
3    112808
4     35991
Name: count, dtype: int64


##Entrenamiento

In [ ]:
LABEL_NAMES = [
    'Sin descuento (<20%)', 'Moderado (20-49%)',
    'Grande (50-74%)','Agresivo (75-89%)', 'Extremo (90%+)'
]

def entrenar_y_guardar(version, params_extra=None):
    params = dict(
        objective='multi:softmax',
        num_class=5,
        n_estimators=500,
        max_depth=6,
        learning_rate=0.01,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        tree_method='hist',
        early_stopping_rounds=20
    )
    if params_extra:
        params.update(params_extra)

    modelo = xgb.XGBClassifier(**params)
    modelo.fit(
        X_train, y_train,
        sample_weight=sample_weights,
        eval_set=[(X_test, y_test)],
        verbose=10
    )

    y_pred= modelo.predict(X_test)
    report= classification_report(y_test, y_pred, target_names=LABEL_NAMES, output_dict=True)
    accuracy= report['accuracy']
    mlogloss= modelo.best_score

    print(f'{version}, iteración {modelo.best_iteration} ===')
    print(classification_report(y_test, y_pred, target_names=LABEL_NAMES))
    print('Matriz de confusión:')
    print(confusion_matrix(y_test, y_pred))

    # Sera util para ver donde cae todo el peso
    importancias = pd.Series(
        modelo.feature_importances_, index=X_train.columns
    ).sort_values(ascending=False)
    print('\nTop 15 features:')
    print(importancias.head(15))

    ts= datetime.now().strftime('%Y%m%d_%H%M')
    filename = f'discount_predictor_v{version}_loss{mlogloss:.4f}_acc{accuracy:.4f}_{ts}.pkl'
    filepath = os.path.join(MODELS_DIR, filename)
    joblib.dump(modelo, filepath)
    print(f'Modelo guardado: {filepath}')

    return modelo, importancias


Función lista.


In [33]:
modelo_v5, imp_v5 = entrenar_y_guardar(version=5)

[0]	validation_0-mlogloss:1.52822
[10]	validation_0-mlogloss:1.17952
[20]	validation_0-mlogloss:1.07751
[30]	validation_0-mlogloss:1.03741
[40]	validation_0-mlogloss:1.03167
[50]	validation_0-mlogloss:1.02345
[60]	validation_0-mlogloss:1.03215
[66]	validation_0-mlogloss:1.03779

=== v5 — iteración 46 ===
                      precision    recall  f1-score   support

Sin descuento (<20%)       0.66      0.38      0.48     46414
   Moderado (20-49%)       0.56      0.57      0.57     11298
     Grande (50-74%)       0.51      0.81      0.62     14280
   Agresivo (75-89%)       0.53      0.79      0.63     18562
      Extremo (90%+)       0.54      0.78      0.64      5321

            accuracy                           0.56     95875
           macro avg       0.56      0.66      0.59     95875
        weighted avg       0.59      0.56      0.55     95875

Matriz de confusión:
[[17441  3817 10130 12147  2879]
 [ 3703  6419   459   398   319]
 [ 1720   654 11518   371    17]
 [ 2675   437

## intento 2

In [ ]:

modelo_v6, imp_v6 = entrenar_y_guardar(
    version=6,
    params_extra={'max_depth': 8, 'learning_rate': 0.05}
)

[0]	validation_0-mlogloss:1.56382
[10]	validation_0-mlogloss:1.29077
[20]	validation_0-mlogloss:1.15335
[30]	validation_0-mlogloss:1.08216
[40]	validation_0-mlogloss:1.04612
[50]	validation_0-mlogloss:1.02935
[60]	validation_0-mlogloss:1.02688
[70]	validation_0-mlogloss:1.02755
[80]	validation_0-mlogloss:1.03343
[84]	validation_0-mlogloss:1.03686

=== v6 — iteración 64 ===
                      precision    recall  f1-score   support

Sin descuento (<20%)       0.66      0.39      0.49     46414
   Moderado (20-49%)       0.56      0.56      0.56     11298
     Grande (50-74%)       0.51      0.80      0.63     14280
   Agresivo (75-89%)       0.53      0.78      0.63     18562
      Extremo (90%+)       0.54      0.79      0.64      5321

            accuracy                           0.57     95875
           macro avg       0.56      0.67      0.59     95875
        weighted avg       0.60      0.57      0.55     95875

Matriz de confusión:
[[18022  3926  9856 11631  2979]
 [ 3624  

## intento 3

In [ ]:
modelo_v7, imp_v7 = entrenar_y_guardar(version=7,params_extra={'max_depth': 5, 'reg_alpha': 0.1, 'reg_lambda': 2.0}
)

[0]	validation_0-mlogloss:1.52984
[10]	validation_0-mlogloss:1.18047
[20]	validation_0-mlogloss:1.08019
[30]	validation_0-mlogloss:1.03917
[40]	validation_0-mlogloss:1.03210
[50]	validation_0-mlogloss:1.03223
[60]	validation_0-mlogloss:1.03824
[62]	validation_0-mlogloss:1.04079

=== v7 — iteración 42 ===
                      precision    recall  f1-score   support

Sin descuento (<20%)       0.66      0.33      0.44     46414
   Moderado (20-49%)       0.55      0.59      0.57     11298
     Grande (50-74%)       0.50      0.81      0.62     14280
   Agresivo (75-89%)       0.51      0.80      0.63     18562
      Extremo (90%+)       0.51      0.84      0.64      5321

            accuracy                           0.55     95875
           macro avg       0.55      0.67      0.58     95875
        weighted avg       0.59      0.55      0.53     95875

Matriz de confusión:
[[15231  4244 10411 12959  3569]
 [ 3518  6624   413   430   313]
 [ 1581   743 11531   409    16]
 [ 2303   449

In [ ]:

modelos_guardados = sorted(
    [f for f in os.listdir(MODELS_DIR) if f.startswith('discount_predictor')],
    reverse=True
)
print('Modelos disponibles:')
for m in modelos_guardados:
    print(f'  {m}')

Modelos disponibles:
  discount_predictor_v7_loss1.0305_acc0.5502_20260407_2048.pkl
  discount_predictor_v6_loss1.0262_acc0.5689_20260407_2048.pkl
  discount_predictor_v5_loss1.0226_acc0.5647_20260407_2048.pkl
